# Task 5 – Level 1: California Housing Price Regression
**IEEE SSCS AI Sub-Team**

---

## Objective
Build and evaluate regression models to predict **Median House Value** using the California Housing Prices dataset (1990 California Census).

## Approach
1. Data Loading & Exploration (EDA)
2. Preprocessing & Feature Scaling
3. Train/Validation/Test Split (70/15/15)
4. Linear Regression from Scratch:
   - Normal Equation
   - Gradient Descent
5. Linear Regression using Scikit-learn
6. Metrics from Scratch (MSE, MAE)
7. Comparison & Analysis

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

# For reproducibility
np.random.seed(42)

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded successfully')

---
## 1. Load & Explore the Dataset

In [ ]:
# Load the dataset
df = pd.read_csv('california_housing.csv')

print(f'Dataset shape: {df.shape}')
print(f'\nFeatures: {df.columns.tolist()}')
df.head()

In [ ]:
# Statistical summary
df.describe().round(2)

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.any() else  ' No missing values found!')

In [ ]:
# Distribution of the target variable
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['median_house_value'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribution of Median House Value', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Median House Value (USD)')
axes[0].set_ylabel('Frequency')

axes[1].hist(np.log1p(df['median_house_value']), bins=50, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title('Log-Transformed Target Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('log(Median House Value + 1)')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.savefig('target_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 9))
corr_matrix = df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Top features correlated with target
target_corr = corr_matrix['median_house_value'].drop('median_house_value').sort_values(key=abs, ascending=False)
print('Feature Correlations with Median House Value:')
print(target_corr.round(4).to_string())

In [ ]:
# Scatter plots: top correlated features vs target
top_features = ['median_income', 'distance_to_coast', 'distance_to_los_angeles', 'housing_median_age']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
colors = ['steelblue', 'coral', 'seagreen', 'mediumpurple']

for ax, feat, color in zip(axes.flatten(), top_features, colors):
    ax.scatter(df[feat], df['median_house_value'], alpha=0.1, s=5, color=color)
    ax.set_xlabel(feat.replace('_', ' ').title(), fontsize=11)
    ax.set_ylabel('Median House Value', fontsize=11)
    ax.set_title(f'{feat.replace("_", " ").title()} vs Target', fontsize=12, fontweight='bold')

plt.suptitle('Feature vs Target Scatter Plots', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 2. Preprocessing

In [ ]:
# Separate features and target
X = df.drop(columns=['median_house_value']).values   # shape: (n, 13)
y = df['median_house_value'].values                  # shape: (n,)
feature_names = df.drop(columns=['median_house_value']).columns.tolist()

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

---
## 3. Train / Validation / Test Split (70% / 15% / 15%)

In [ ]:
def train_val_test_split(X, y, train_ratio=0.70, val_ratio=0.15, seed=42):
    """
    Split dataset into training, validation, and test sets.
    
    Parameters:
    - X: feature matrix
    - y: target vector
    - train_ratio: fraction for training
    - val_ratio:   fraction for validation (test = 1 - train - val)
    - seed: random seed for reproducibility
    
    Returns: X_train, X_val, X_test, y_train, y_val, y_test
    """
    rng = np.random.default_rng(seed)
    n = len(X)
    
    # Shuffle indices
    indices = rng.permutation(n)
    
    # Compute split points
    train_end = int(n * train_ratio)
    val_end   = int(n * (train_ratio + val_ratio))
    
    train_idx = indices[:train_end]
    val_idx   = indices[train_end:val_end]
    test_idx  = indices[val_end:]
    
    return (X[train_idx], X[val_idx], X[test_idx],
            y[train_idx], y[val_idx], y[test_idx])


X_train, X_val, X_test, y_train, y_val, y_test = train_val_test_split(X, y)

print(f'Training set   : {X_train.shape[0]} samples  ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation set : {X_val.shape[0]} samples  ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test set       : {X_test.shape[0]} samples  ({X_test.shape[0]/len(X)*100:.1f}%)')

### Feature Scaling (StandardScaler from scratch)
>  **Important:** We fit the scaler **only on training data**, then apply the same parameters to val/test to prevent **data leakage**.

In [ ]:
class StandardScalerScratch:
    """
    Standard Scaler from scratch.
    Transforms features to have zero mean and unit variance: z = (x - μ) / σ
    Fit ONLY on training data, then transform all sets.
    """
    def fit(self, X):
        self.mean_ = X.mean(axis=0)
        self.std_  = X.std(axis=0)
        # Avoid division by zero for constant features
        self.std_[self.std_ == 0] = 1.0
        return self
    
    def transform(self, X):
        return (X - self.mean_) / self.std_
    
    def fit_transform(self, X):
        return self.fit(X).transform(X)


scaler = StandardScalerScratch()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

print('Feature scaling done.')
print(f'Training mean  (first 3 features): {X_train_scaled[:, :3].mean(axis=0).round(6)}')
print(f'Training std   (first 3 features): {X_train_scaled[:, :3].std(axis=0).round(6)}')

---
## 4. Metrics from Scratch
> Both MSE and MAE are implemented using basic NumPy operations only.

In [ ]:
def mse_scratch(y_true, y_pred):
    """
    Mean Squared Error (MSE) from scratch.
    MSE = (1/n) * Σ(y_true - y_pred)²
    """
    errors = y_true - y_pred
    return np.sum(errors ** 2) / len(y_true)


def mae_scratch(y_true, y_pred):
    """
    Mean Absolute Error (MAE) from scratch.
    MAE = (1/n) * Σ|y_true - y_pred|
    """
    return np.sum(np.abs(y_true - y_pred)) / len(y_true)


def rmse_scratch(y_true, y_pred):
    """Root Mean Squared Error for better interpretability."""
    return np.sqrt(mse_scratch(y_true, y_pred))


def r2_scratch(y_true, y_pred):
    """
    R² (Coefficient of Determination) from scratch.
    R² = 1 - SS_res / SS_tot
    """
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return 1 - (ss_res / ss_tot)


print(' Metric functions defined: MSE, MAE, RMSE, R²')

---
## 5. Model 1 – Normal Equation (Closed-Form Solution)

$$w = (X^TX)^{-1}X^T y$$

This gives the **exact** optimal solution analytically — no iterations needed.

In [ ]:
class NormalEquationRegression:
    """
    Linear Regression via the Normal Equation (closed-form solution).
    
    Derivation:
    Minimize J(w) = (1/2n)||Xw - y||²
    Setting gradient = 0 → w = (XᵀX)⁻¹ Xᵀ y
    
    Note: Adds bias column (ones) to X internally.
    Note: Uses np.linalg.pinv (pseudo-inverse) for numerical stability
          in case XᵀX is singular or near-singular.
    """
    
    def fit(self, X, y):
        # Add bias term: augment X with a column of ones → X_b shape: (n, p+1)
        n = X.shape[0]
        X_b = np.hstack([np.ones((n, 1)), X])     # bias column prepended
        
        # Normal Equation: w = (XᵀX)⁻¹ Xᵀ y
        # Using pseudo-inverse for robustness (handles singular matrices)
        self.w_ = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        
        return self
    
    def predict(self, X):
        n = X.shape[0]
        X_b = np.hstack([np.ones((n, 1)), X])
        return X_b @ self.w_
    
    @property
    def bias(self):
        return self.w_[0]
    
    @property
    def coef(self):
        return self.w_[1:]


# Train the model
ne_model = NormalEquationRegression()
ne_model.fit(X_train_scaled, y_train)

# Predict on all splits
ne_pred_train = ne_model.predict(X_train_scaled)
ne_pred_val   = ne_model.predict(X_val_scaled)
ne_pred_test  = ne_model.predict(X_test_scaled)

print('=' * 55)
print('        Normal Equation – Performance')
print('=' * 55)
print(f'{"Split":<12} {"MSE":>15} {"MAE":>12} {"R²":>8}')
print('-' * 55)
for split, y_true, y_pred in [
    ('Train',      y_train, ne_pred_train),
    ('Validation', y_val,   ne_pred_val),
    ('Test',       y_test,  ne_pred_test),
]:
    mse = mse_scratch(y_true, y_pred)
    mae = mae_scratch(y_true, y_pred)
    r2  = r2_scratch(y_true, y_pred)
    print(f'{split:<12} {mse:>15,.2f} {mae:>12,.2f} {r2:>8.4f}')
print('=' * 55)

In [ ]:
# Feature importance from Normal Equation weights
coef_df = pd.DataFrame({
    'Feature': feature_names,
    'Weight': ne_model.coef
}).sort_values('Weight', key=abs, ascending=True)

plt.figure(figsize=(10, 6))
colors = ['coral' if w < 0 else 'steelblue' for w in coef_df['Weight']]
bars = plt.barh(coef_df['Feature'], coef_df['Weight'], color=colors, edgecolor='white')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Normal Equation – Feature Weights', fontsize=13, fontweight='bold')
plt.xlabel('Weight Value')
plt.tight_layout()
plt.savefig('ne_feature_weights.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Model 2 – Gradient Descent

**Update rule:**
$$w := w - \alpha \nabla J(w)$$

Where the gradient of MSE is:
$$\nabla J(w) = \frac{1}{n} X^T (Xw - y)$$

In [ ]:
class GradientDescentRegression:
    """
    Linear Regression via Batch Gradient Descent.
    
    Cost function (MSE):
        J(w) = (1/2n) * ||Xw - y||²
    
    Gradient:
        ∇J(w) = (1/n) * Xᵀ(Xw - y)
    
    Update rule:
        w := w - α * ∇J(w)
    
    Parameters:
    - learning_rate (α): step size per iteration
    - n_iterations:  max number of iterations
    - tol:           convergence tolerance (stops early if Δloss < tol)
    """
    
    def __init__(self, learning_rate=0.01, n_iterations=1000, tol=1e-6):
        self.lr     = learning_rate
        self.n_iter = n_iterations
        self.tol    = tol
    
    def fit(self, X, y):
        n, p = X.shape
        
        # Add bias column
        X_b = np.hstack([np.ones((n, 1)), X])   # shape: (n, p+1)
        
        # Initialize weights to zeros
        self.w_ = np.zeros(p + 1)               # shape: (p+1,)
        
        self.loss_history_ = []
        
        for iteration in range(self.n_iter):
            # Forward pass: compute predictions
            y_pred = X_b @ self.w_               # shape: (n,)
            
            # Compute residuals
            residuals = y_pred - y               # shape: (n,)
            
            # Compute MSE loss (for tracking)
            loss = np.sum(residuals ** 2) / (2 * n)
            self.loss_history_.append(loss)
            
            # Early stopping: check convergence
            if iteration > 0 and abs(self.loss_history_[-2] - loss) < self.tol:
                print(f' Converged at iteration {iteration}')
                break
            
            # Compute gradient: ∇J = (1/n) * Xᵀ * residuals
            gradient = (X_b.T @ residuals) / n  # shape: (p+1,)
            
            # Update weights: w := w - α * ∇J
            self.w_ -= self.lr * gradient
        
        else:
            print(f' Reached max iterations ({self.n_iter})')
        
        return self
    
    def predict(self, X):
        n = X.shape[0]
        X_b = np.hstack([np.ones((n, 1)), X])
        return X_b @ self.w_
    
    @property
    def bias(self):
        return self.w_[0]
    
    @property
    def coef(self):
        return self.w_[1:]


# Train Gradient Descent model
# (Feature scaling was critical for GD convergence — we already did it above)
gd_model = GradientDescentRegression(learning_rate=0.1, n_iterations=2000, tol=1e-8)
gd_model.fit(X_train_scaled, y_train)

# Predict on all splits
gd_pred_train = gd_model.predict(X_train_scaled)
gd_pred_val   = gd_model.predict(X_val_scaled)
gd_pred_test  = gd_model.predict(X_test_scaled)

print()
print('=' * 55)
print('        Gradient Descent – Performance')
print('=' * 55)
print(f'{"Split":<12} {"MSE":>15} {"MAE":>12} {"R²":>8}')
print('-' * 55)
for split, y_true, y_pred in [
    ('Train',      y_train, gd_pred_train),
    ('Validation', y_val,   gd_pred_val),
    ('Test',       y_test,  gd_pred_test),
]:
    mse = mse_scratch(y_true, y_pred)
    mae = mae_scratch(y_true, y_pred)
    r2  = r2_scratch(y_true, y_pred)
    print(f'{split:<12} {mse:>15,.2f} {mae:>12,.2f} {r2:>8.4f}')
print('=' * 55)

In [ ]:
# Plot loss curve
plt.figure(figsize=(10, 5))
plt.plot(gd_model.loss_history_, color='steelblue', linewidth=2)
plt.title('Gradient Descent – Loss Curve (MSE/2)', fontsize=13, fontweight='bold')
plt.xlabel('Iteration')
plt.ylabel('Loss')
plt.yscale('log')
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('gd_loss_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Initial loss  : {gd_model.loss_history_[0]:,.2f}')
print(f'Final loss    : {gd_model.loss_history_[-1]:,.2f}')
print(f'Total iters   : {len(gd_model.loss_history_)}')

---
## 7. Model 3 – Scikit-learn Linear Regression

In [ ]:
# Scikit-learn uses a highly optimized solver (similar to Normal Equation + SVD decomposition)
sklearn_model = LinearRegression()
sklearn_model.fit(X_train_scaled, y_train)

# Predict
sk_pred_train = sklearn_model.predict(X_train_scaled)
sk_pred_val   = sklearn_model.predict(X_val_scaled)
sk_pred_test  = sklearn_model.predict(X_test_scaled)

print('=' * 55)
print('        Scikit-learn – Performance')
print('=' * 55)
print(f'{"Split":<12} {"MSE":>15} {"MAE":>12} {"R²":>8}')
print('-' * 55)
for split, y_true, y_pred in [
    ('Train',      y_train, sk_pred_train),
    ('Validation', y_val,   sk_pred_val),
    ('Test',       y_test,  sk_pred_test),
]:
    mse = mse_scratch(y_true, y_pred)
    mae = mae_scratch(y_true, y_pred)
    r2  = r2_scratch(y_true, y_pred)
    print(f'{split:<12} {mse:>15,.2f} {mae:>12,.2f} {r2:>8.4f}')
print('=' * 55)

---
## 8. Comprehensive Model Comparison

In [ ]:
# Build results table for all models on the TEST set
models = {
    'Normal Equation (Scratch)': ne_pred_test,
    'Gradient Descent (Scratch)': gd_pred_test,
    'Scikit-learn LinearReg':    sk_pred_test,
}

results = []
for name, y_pred in models.items():
    results.append({
        'Model': name,
        'MSE': mse_scratch(y_test, y_pred),
        'MAE': mae_scratch(y_test, y_pred),
        'RMSE': rmse_scratch(y_test, y_pred),
        'R²': r2_scratch(y_test, y_pred),
    })

results_df = pd.DataFrame(results).set_index('Model')
print(' TEST SET – All Models Comparison')
print(results_df.round(2).to_string())

In [ ]:
# Visual comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
model_labels = ['Normal\nEquation', 'Gradient\nDescent', 'Scikit-learn']
bar_colors   = ['steelblue', 'coral', 'seagreen']

metrics = [
    ('MSE',  'MSE (lower = better)'),
    ('MAE',  'MAE (lower = better)'),
    ('R²',   'R² (higher = better)'),
]

for ax, (metric, title) in zip(axes, metrics):
    vals = results_df[metric].values
    bars = ax.bar(model_labels, vals, color=bar_colors, edgecolor='white', width=0.5)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel(metric)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.01,
                f'{val:,.1f}' if metric != 'R²' else f'{val:.4f}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('Model Comparison on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Actual vs Predicted plots for all models
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
model_preds = [
    ('Normal Equation', ne_pred_test, 'steelblue'),
    ('Gradient Descent', gd_pred_test, 'coral'),
    ('Scikit-learn', sk_pred_test, 'seagreen'),
]

for ax, (name, y_pred, color) in zip(axes, model_preds):
    ax.scatter(y_test, y_pred, alpha=0.2, s=5, color=color)
    # Perfect prediction line
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, 'k--', linewidth=1.5, label='Perfect fit')
    ax.set_title(f'{name}\nR²={r2_scratch(y_test, y_pred):.4f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual Values')
    ax.set_ylabel('Predicted Values')
    ax.legend(fontsize=8)

plt.suptitle('Actual vs Predicted – Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, (name, y_pred, color) in zip(axes, model_preds):
    residuals = y_test - y_pred
    ax.scatter(y_pred, residuals, alpha=0.2, s=5, color=color)
    ax.axhline(0, color='black', linewidth=1.2, linestyle='--')
    ax.set_title(f'{name} – Residuals', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Values')
    ax.set_ylabel('Residuals')

plt.suptitle('Residual Plots – Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('residual_plots.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Weight comparison between scratch methods and sklearn
coef_comparison = pd.DataFrame({
    'Feature': feature_names,
    'Normal Equation': ne_model.coef,
    'Gradient Descent': gd_model.coef,
    'Scikit-learn': sklearn_model.coef_,
}).set_index('Feature')

print(' Weight Comparison Across Models:')
print(coef_comparison.round(4).to_string())
print(f'\nBias (intercept):')
print(f'  Normal Equation : {ne_model.bias:,.2f}')
print(f'  Gradient Descent: {gd_model.bias:,.2f}')
print(f'  Scikit-learn    : {sklearn_model.intercept_:,.2f}')

---
## 9. Discussion & Comparison

---

###  Normal Equation vs Gradient Descent

| Aspect | Normal Equation | Gradient Descent |
|--------|----------------|------------------|
| **Solution type** | Exact, closed-form | Iterative approximation |
| **Convergence** | One step (no iterations) | Requires tuning α and many iterations |
| **Feature scaling** | Not required | **Required** for fast convergence |
| **Computational cost** | O(p³) — expensive for large p | O(n·p) per iteration — scalable |
| **Memory** | Must invert p×p matrix | Only needs gradient vector |
| **Large datasets** | Impractical (n > 100k) | Preferred (use mini-batch) |
| **Result quality** | Globally optimal | Optimal if converged + convex |

###  Why do all three models give nearly identical results?

Linear Regression has a **convex loss surface** — there is exactly **one global minimum**. Whether you solve analytically (Normal Equation) or iteratively (Gradient Descent), both converge to the same weights. Scikit-learn also uses an efficient closed-form solver (via SVD decomposition), so all three should give matching predictions.

###  Key Observations

- **Median Income** has the strongest positive correlation with house value — the higher the income, the higher the property price.
- **Distance to Coast** has a negative impact — properties farther from the coast are cheaper.
- **Feature scaling is critical for Gradient Descent**: without it, features with large magnitudes (e.g., population = millions vs income = tens of thousands) cause slow or divergent training.
- Small gap between Normal Equation and Gradient Descent results indicates that GD **converged successfully** to the optimal solution.
- R² ≈ 0.6–0.7 is reasonable for Linear Regression on this dataset. The relationship between features and house price is **not purely linear** — tree-based models (Random Forest, XGBoost) would significantly improve performance.

###  Next Steps for Improvement
- Feature engineering: add interaction terms, log-transform skewed features
- Try Ridge / Lasso Regression to reduce overfitting
- Use polynomial features to capture non-linear relationships
- Try non-linear models: Decision Trees, Random Forest, XGBoost

In [ ]:
print('='*60)
print('           FINAL SUMMARY – TEST SET RESULTS')
print('='*60)
print(f'{"Model":<30} {"MSE":>12} {"MAE":>10} {"R²":>8}')
print('-'*60)
for idx, row in results_df.iterrows():
    print(f'{idx:<30} {row["MSE"]:>12,.2f} {row["MAE"]:>10,.2f} {row["R²"]:>8.4f}')
print('='*60)
print('\n Task 5 Complete! All models trained and evaluated.')